# Expérience D : occurrence A et croissance bâtiment → global

Une seule variante, **alpha = 30 fixé avant calcul des résultats**. Ni grille, ni recherche de poids, ni système adaptatif, ni forecast 2026. A/B et la fondation restent intacts. [Protocole et rapport détaillé](../docs/model_d_experiment.md).

`w_b = n_b/(n_b+30)` : 30 observations donnent 50 % de poids au bâtiment, 10 en donnent 25 %. Rappel conservateur vers le global, sans choix d'alpha sur le MAE final.

Train : exactement les unités/années positives qualifiées de A ; positive = occurrence 1, **pas croissance > 0**. Les croissances négatives restent incluses. Bâtiment connu à sa propre origine historique ; labels mûrs au cutoff. La sécurité reste conditionnelle à la fidélité CRM, sans versions archivées.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), 'Exécuter depuis le dépôt ou notebooks/.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.models.model_d import ALPHA
from src.evaluation.model_d_evaluation import run_model_d_experiment
assert ALPHA == 30, 'Le protocole alpha est figé.'
leases = pd.read_csv(ROOT / 'data/raw/equinoxe_lease_history.csv',
                     dtype={'sPropCode':'string', 'sUnitCode':'string'})
# Les baux et prédictions unitaires restent en mémoire, sans affichage/export.


## Calcul à cutoff et contrôles communs

Les trois forecasts D sont fixés avant scoring. `p_i` est exactement celui de A ; seul g change. `P1_D = 100 × médiane(p_i*g_i)`. Les mêmes cohortes et mêmes labels réalisés que A sont vérifiés. Les comparateurs figés sont recalculés sans modification ni sélection de champion.

In [2]:
experiment = run_model_d_experiment(leases)
for name, table in experiment.items():
    assert not {'UNIT_KEY','sUnitCode','sPropCode','hTenant','hLease'}.intersection(table.columns), name
display(experiment['checks'].round(6))

,year,prediction_origin,candidate_units,observed_units,train_positive_unit_years,alpha,identical_A_occurrence,identical_A_cohort_and_target,g_global_pct,temporal_status
0,2023,2022-12-31,405,401,882,30.0,True,True,2.918870,SAFE_WITH_CONDITIONS
1,2024,2023-12-31,645,634,1227,30.0,True,True,3.319802,SAFE_WITH_CONDITIONS
2,2025,2024-12-31,856,773,1627,30.0,True,True,3.024079,SAFE_WITH_CONDITIONS


## Prévisions, erreurs signées et absolues

Prévisions et réalisés en %, erreurs en points de pourcentage. B représente **Model B — full hierarchical**, P1_G1 ; 50/50 reste la moyenne fixe des forecasts scalaires.

In [3]:
display(experiment['results'].round(6))
display(experiment['metrics'].round(6))

,year,model,prediction_pct,realized_pct,error_pp,absolute_error_pp
0,2023,A,2.870060,2.036102,0.833957,0.833957
1,2023,Model B — full hierarchical,2.871314,2.036102,0.835212,0.835212
2,2023,A/B 50-50,2.870687,2.036102,0.834585,0.834585
3,2023,D,2.870927,2.036102,0.834825,0.834825
4,2024,A,3.269179,1.607852,1.661326,1.661326
5,2024,Model B — full hierarchical,3.236099,1.607852,1.628247,1.628247
6,2024,A/B 50-50,3.252639,1.607852,1.644787,1.644787
7,2024,D,3.289002,1.607852,1.681150,1.681150
8,2025,A,2.969328,3.300878,-0.331551,0.331551
9,2025,Model B — full hierarchical,2.924144,3.300878,-0.376735,0.376735


,model,MAE_pp,RMSE_pp,bias_pp,worst_absolute_error_pp
0,A,0.942278,1.090171,0.721244,1.661326
1,Model B — full hierarchical,0.946731,1.078687,0.695575,1.628247
2,A/B 50-50,0.944505,1.084323,0.708410,1.644787
3,D,0.949175,1.100473,0.728141,1.681150


## Diagnostics bâtiment et composition

Chaque ligne est un agrégat : n_b compte des unités/années positives qualifiées. Les statistiques historiques sont masquées sous cinq observations ; sans historique, repli au global. Une part de cohorte n'est pas une contribution additive au P1 : la médiane n'est pas linéaire.

In [4]:
display(experiment['building_diagnostics'].round(6))
composition = experiment['building_diagnostics'].groupby('year').agg(
    cohort=('candidate_units','sum'), share_pct=('cohort_share_pct','sum'))
assert composition.cohort.tolist() == [405,645,856]
assert composition.share_pct.round(10).eq(100).all()
display(composition)

,year,building,candidate_units,cohort_share_pct,n_b,w_b,g_global_pct,g_b_pct,g_b_shrunk_pct,fallback_global,history_statistics_masked
0,2023,Daniel-Johnson,111,27.407407,96,0.761905,2.918870,3.400080,3.285506,False,False
1,2023,Levesque,81,20.000000,221,0.880478,2.918870,2.165960,2.255949,False,False
2,2023,Saint-Elzear,213,52.592593,565,0.949580,2.918870,2.919799,2.919752,False,False
3,2024,Daniel-Johnson,125,19.379845,153,0.836066,3.319802,3.817923,3.736264,False,False
4,2024,Le Carlyle,127,19.689922,0,0.000000,3.319802,NaN,3.319802,True,False
5,2024,Levesque,82,12.713178,300,0.909091,3.319802,2.712845,2.768023,False,False
6,2024,Saint-Elzear,215,33.333333,774,0.962687,3.319802,3.340712,3.339932,False,False
7,2024,The Met,96,14.883721,0,0.000000,3.319802,NaN,3.319802,True,False
8,2025,Daniel-Johnson,129,15.070093,262,0.897260,3.024079,3.314405,3.284577,False,False
9,2025,Le Carlyle,180,21.028037,0,0.000000,3.024079,NaN,3.024079,True,False


,cohort,share_pct
year,,
2023,405,100.0
2024,645,100.0
2025,856,100.0


## Explication des résultats

- **2023** : Saint-Elzear représente 213/405 candidates et porte la médiane ; son g shrinké dépasse très légèrement le global. D dégrade l'erreur absolue de **0,000868 pp**.
- **2024** : 82 Levesque sous le global + 223 candidates Le Carlyle/The Met au repli global = 305 candidates ; la position médiane 323 appartient encore à Saint-Elzear. Son g shrinké **3,339932 %** dépasse le global **3,319802 %** : D dégrade l'erreur de **0,019824 pp**.
- **2025** : 429 candidates sans historique positif qualifié se replient au global. Avec 82 Levesque en dessous, ce bloc couvre les deux positions médianes 428/429. D reproduit exactement A ; l'occurrence de Saint-Elzear reste inchangée.

La composition bâtiment influe donc sur l'estimation, sans apporter de gain observé. Aucun poids ou alpha n'est changé après ces constats.

## Réponses aux huit questions

1. **Améliore 2024 ?** Non : erreur absolue 1,681150 pp, contre A 1,661326.
2. **Dégrade 2023 ?** Oui, très légèrement : 0,834825 contre 0,833957 pp.
3. **Dégrade 2025 ?** Non : erreur identique −0,331551 pp, sans correction de l'occurrence.
4. **Gain par composition ?** Aucun gain ; les rangs de la médiane expliquent la hausse 2024 et l'identité 2025.
5. **Stabilité/complexité justifiées ?** Aucun backtest amélioré, toutes les métriques agrégées inférieures à A : non justifiées ici.
6. **Information différente de B ?** Segmentation et train différents, mais aucune complémentarité P1 validée ; mêmes signes d'erreur.
7. **Disponible au cutoff ?** Helpers datés et maturité identiques à A ; sécurité conditionnelle à la fidélité historique CRM. Disponibilité archivée non démontrée, sans nouveau signal rétrospectif.
8. **Risque de sélection ?** Oui, hypothèse formulée après les mêmes trois années ; alpha fixe sans tuning ne crée pas un holdout indépendant.

## Recommandation

**REJECT_D_RETURN_TO_A_B** : D n'améliore aucune année et aggrave 2024. Aucun autre alpha testé, aucun champion final sélectionné, aucune nouvelle hiérarchie, aucun système adaptatif, **aucun forecast 2026**. STOP après cette expérience.